In [ ]:
from selenium import webdriver
from selenium.webdriver import Chrome, ChromeOptions
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import  WebDriverWait
import pandas as pd 
from selenium.webdriver.common.action_chains import ActionChains
import time, threading, os, requests, ast,json,re
from datetime import datetime


def GetAuctionDetails(driver):
    wait = WebDriverWait(driver, 15)

    try:
        # elements
        auction_name = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h1.cc-auction-overview__title"))
        ).text.strip()

        auction_time_raw = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".cc-auction-overview__when dd"))
        ).text.strip()

        auction_type = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".cc-key-pair__definition"))
        ).text.strip()

        center_location = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".cc-auction-overview__where .u-line-height-1"))
        ).text.strip()



        start_part, end_part = auction_time_raw.split(" - ")

        current_year = datetime.now().year


        start_clean = re.sub(r"^\w+\s+", "", start_part).replace(" at ", " ")
        start_dt = datetime.strptime(f"{start_clean} {current_year}", "%d %B %I:%M%p %Y")

        end_clean = re.sub(r"^\w+\s+", "", end_part).replace(" at ", " ")
        end_dt = datetime.strptime(f"{end_clean} {current_year}", "%d %b %I:%M%p %Y")

        auction_data = {
            "auction_name": auction_name,
            "auction_type": auction_type,
            "start_date": start_dt.strftime("%Y-%m-%d"),
            "start_time": start_dt.strftime("%H:%M"),
            "end_date": end_dt.strftime("%Y-%m-%d"),
            "end_time": end_dt.strftime("%H:%M"),
            "center_location": center_location
        }

        with open("auction_details.json", "w", encoding="utf-8") as f:
            json.dump(auction_data, f, indent=4)

        print("✅ Auction details saved")
        print(json.dumps(auction_data, indent=4))

        return auction_data

    except Exception as e:
        print("❌ Error in GetAuctionDetails:", e)
        return None

def scarpe(id):
    path = f"https://www.wilsonsauctions.com/auctions/{id}"
    options = ChromeOptions()
    options.headless = True
    service = Service(ChromeDriverManager().install())
    driver = Chrome(service=service, options=options)
    driver.get(path)
    driver.maximize_window()

    wait = WebDriverWait(driver, 10)
    try:
        cookie_btn = wait.until(EC.element_to_be_clickable((By.ID, 'cookiescript_accept')))
        cookie_btn.click()
  
    except:
        print("No cookies button found")


    try:
        login_btn = wait.until(EC.element_to_be_clickable((By.ID, 'gtm-sign-in-sign-up')))
        login_btn.click()
    
    except:
        print("Login button not found")


    try:
        username_input = wait.until(EC.presence_of_element_located((By.ID, 'email')))
        username_input.send_keys("fourbrotherstrading@icloud.com")
        next_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@class='c-button']")))
        next_btn.click()

    except:
        print("Username input not found")

    try:
        password_input = wait.until(EC.visibility_of_element_located((By.ID, 'password')))
        password_input.clear()
        password_input.send_keys("Muhssan7865@")  
        login_btn2 = wait.until(
            EC.element_to_be_clickable((By.XPATH, "//button[@class='c-button' and @data-label='log-in']"))
        )
        login_btn2.click()

    except Exception as e:
        print("Password input not found or error:", e)
    GetAuctionDetails(driver)
    try:
        first_lot = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//ul[@class='cc-cards +list +results u-marg-top u-pad-top']//li[1]//a")
            )
        )
        lot_url = first_lot.get_attribute("href")
        print(f"Opening first lot: {lot_url}")
        driver.execute_script("arguments[0].click();", first_lot)
    except Exception as e:
        print("Could not open first lot:", e)
        
    folder_name = "timehtml"
    os.makedirs(folder_name, exist_ok=True)

    lot_number = 1

    while True:
        try:
            get_Reg = wait.until(
                EC.presence_of_element_located(
                    (By.XPATH, "//div[@class='cc-reg-plate__end']//span")
                )
            )
            reg_number = get_Reg.text.strip()
            print(f"🚗 Registration Number: {reg_number}")

            # 🔹 OPEN INSPECTION TAB
            try:
                inspect_tab = wait.until(
                    EC.element_to_be_clickable((
                        By.XPATH,
                        "//button[contains(., 'Inspection')] | //a[contains(., 'Inspection')]"
                    ))
                )
                driver.execute_script("arguments[0].click();", inspect_tab)
                time.sleep(2)
            except:
                pass

            # 🔹 EXPAND ALL SECTIONS (if any)
            try:
                expands = driver.find_elements(
                    By.XPATH, "//button[contains(@class,'accordion')]"
                )
                for e in expands:
                    driver.execute_script("arguments[0].click();", e)
                    time.sleep(0.3)
            except:
                pass

            # 🔹 SAVE FULL HTML
            with open(f"{folder_name}/{reg_number}.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)

            print(f"✅ Saved inspection HTML: {reg_number}.html")

            # 🔹 NEXT LOT

            next_btn = wait.until(
                EC.element_to_be_clickable((
                    By.XPATH,
                    "//button[.//span[contains(normalize-space(), 'Next')]]"
                ))
            )
            print(next_btn)
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(2)

            lot_number += 1

        except Exception as e:
            print("🛑 No more lots:", e)
            break

scarpe("seized-vehicle-auction-2686")
        

✅ Auction details saved
{
    "auction_name": "Seized Vehicle Auction",
    "auction_type": "Timed Online Auction\nLearn more",
    "start_date": "2026-01-21",
    "start_time": "12:00",
    "end_date": "2026-01-22",
    "end_time": "12:00",
    "center_location": "Lots located in multiple locations\nSee individual lots for more details"
}
Opening first lot: https://www.wilsonsauctions.com/auctions/seized-vehicle-auction-2686/lots/237358
🚗 Registration Number: WG61FFJ
🛑 No more lots: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x10a1213
	0x10a1254
	0xe8e52b
	0xecc635
	0xefb3a6
	0xef6ed1
	0xef6846
	0xe5ebfd
	0xe5f18e
	0xe5f65d
	0x12f5254
	0x12f080b
	0x130d0ea
	0x10bb118
	0x10c311d
	0xe5e7ab
	0xe5ddf7
	0x14416ef
	0x759c5d49
	0x76ecd5db
	0x76ecd561



In [10]:
import os,re,json
import csv
from bs4 import BeautifulSoup
from datetime import datetime
with open(r"D:\bots\timeheader.json", "r", encoding="utf-8") as f:
    header_map = json.load(f)
headers = [header_map[k] for k in sorted(header_map, key=int)]

with open("auction_details.json","r",encoding="utf-8") as f:
    database = json.load(f)

def get_base_folder_info():
    folder_name = os.path.basename(os.getcwd())

    parts = folder_name.split("-")
    if not parts or not parts[0].isdigit():
        return None, None

    sheet_id = parts[0]
    name_parts = parts[1:]
    if name_parts and name_parts[0].isdigit():
        name_parts = name_parts[1:]

    auction_name = "-".join(name_parts).strip()


    return sheet_id, auction_name


def yearGetter(val):
    if not val:
        return ""
    match = re.search(r"(\d{4})", val)
    if match:
        return match.group(1)
    return ""

def gettitle(soup):
    title_el = soup.find("h1", class_="c-heading +h1")
    title_text = title_el.get_text(strip=True) if title_el else "No Title Found"
    return title_text


def extract_make_model(title):
    if not title:
        return "", ""

    parts = title.split()

    make = parts[0]
    model = " ".join(parts[1:3])

    return make, model

def getDeraivative(soup):
    p = soup.find("section", class_="cc-lot-summary__desc")
    if not p:
        return ""

    text = p.find("p").get_text(strip=True)
    main = text.split("/")[0].strip()
    parts = main.split()
    derivative = " ".join(parts[1:])  

    return derivative

def extract_key_value_details(soup):
    data = {}

    rows = soup.select("dl.cc-key-pairs .cc-key-pair")

    for row in rows:
        key = row.find("dt")
        value = row.find("dd")

        if key and value:
            k = key.get_text(strip=True)
            v = value.get_text(strip=True)
            data[k] = v

    return data

def extract_images(soup):
    images = []

    for img in soup.select(".swiper-slide img"):
        url = img.get("src")  
        if url:
            images.append(url)

  
    images = list(dict.fromkeys(images))


    return ",".join(images)


def clean_auction_type(text):
    if not text:
        return ""

    text = text.replace("\n", " ").strip()
    text = text.replace("Online", "").replace("Learn more", "").strip()
    text = " ".join(text.split())

    return text
def clean_value(val):
    if not val:
        return ""

    val = str(val).strip()

    if val in ["-", "—", "–", "N―"]:
        return ""

    return val

def engine_size_to_liter(val):
    if not val:
        return ""
    
    match = re.search(r"(\d+)", val.replace(',', ''))
    if not match:
        return ""
    
    cc = int(match.group(1))
    liters = cc / 1000  
    return str(round(liters, 1)) 


def clean_odometer(val):
    if not val:
        return ""
    val = str(val).strip()
    if val in ["-", "—", "–", "N―"]:
        return ""
    
    digits = "".join(c for c in val if c.isdigit())
    return digits


def get_lot_number(soup):
    el = soup.find("div", class_="cc-lot-info")
    if not el:
        return ""

    text = el.get_text(" ", strip=True)

    match = re.search(r"LOT\s+(\d+)", text, re.I)
    return match.group(1) if match else ""


def extract_bid_info(soup):
    result = {"number_of_bids": 0, "current_bid": 0, "deposit_required": 0}

  
    bids_el = soup.find("dt", {"data-qa": "currentBidCount"})
    if bids_el:
        match = re.search(r"\((\d+)\s+Bids\)", bids_el.text)
        if match:
            result["number_of_bids"] = int(match.group(1))

    bid_el = soup.find("dd", class_="cc-stat__detail")
    if bid_el:
        bid_text = bid_el.text.strip().replace("£", "").replace(",", "")
        if bid_text.isdigit():
            result["current_bid"] = int(bid_text)
        else:
            try:
                result["current_bid"] = float(bid_text)
            except:
                result["current_bid"] = 0

 
    deposit_el = soup.find("span", class_="c-alert__text")
    if deposit_el:
        match = re.search(r"Deposit Required:\s*£(\d+)", deposit_el.text)
        if match:
            result["deposit_required"] = int(match.group(1))

    return result

def extract_manual_keys():
    folder = "timehtml"
    output_file = "Final_Time_Wilson.csv"

    all_rows = []

    for file in os.listdir(folder):
        if file.endswith(".html"):
            file_path = os.path.join(folder, file)
            with open(file_path, "r", encoding="utf-8") as f:
                html_content = f.read()
                soup = BeautifulSoup(html_content, "html.parser")

            row = {}

            reg_el = soup.find("div", class_="cc-reg-plate__end")
            regspan = reg_el.find("span") if reg_el else None
            reg = regspan.get_text(strip=True) if regspan else ""

            pattern = re.compile(r'^[A-Z]{1,3}[0-9]{1,3}[A-Z]{1,3}$', re.I)
            if not pattern.match(reg):
                print(f"❌ Not valid: {reg} → Deleting file {file}")
                os.remove(file_path)
                continue
            else:
                result = {}
                lot_loc_el = soup.find("dt", string=lambda x: x and "Lot location" in x)
                if lot_loc_el:
                    dd = lot_loc_el.find_next_sibling("dd")
                    if dd:
 
                        a_tag = dd.find("a")
                        result["location"] = a_tag.text.strip() if a_tag else dd.text.strip()
                    else:
                        result["location"] = ""
                else:
                    result["location"] = ""

                vat_el = soup.find("dt", string=lambda x: x and "Vat status" in x)
                if vat_el:
                    dd = vat_el.find_next_sibling("dd")
                    result["vat_status"] = dd.text.strip() if dd else ""
                else:
                    result["vat_status"] = ""


                
                title =gettitle(soup)
                make, model = extract_make_model(title)
                derivative = getDeraivative(soup)
                sheet_id, auction_name = get_base_folder_info()
                lot_number = get_lot_number(soup)
                auctionName = database.get("auction_name", "")
                auction_type = clean_auction_type(database.get("auction_type", ""))
                start_date = database.get("start_date", "")
                start_time = database.get("start_time", "")
                end_date = database.get("end_date", "")
                end_time = database.get("end_time", "")
                details = extract_key_value_details(soup)
                year = yearGetter(details.get("First reg", ""))
                datas = extract_bid_info(soup)
                Images= extract_images(soup)
                
                print(Images)
        
                
                
                row[header_map['1']] = auctionName or ""
                row[header_map['2']] = sheet_id or ""
                row[header_map['3']] =  "Wilsons Auctions"
                row[header_map['4']] = title
                row[header_map['5']] = reg
                row[header_map['6']] = make
                row[header_map['7']] = model
                row[header_map['8']] = derivative
                row[header_map['9']] = lot_number or ""
                row[header_map['17']] = year or ""
                row[header_map['10']] = clean_value(details.get("Type"))
                row[header_map['24']] = clean_value(details.get("Vendor"))
                row[header_map['41']] = clean_value(details.get("Doors"))
                row[header_map['12']] = clean_value(details.get("Transmission"))
                row[header_map['33']] = clean_value(details.get("Colour"))
                row[header_map['11']] = clean_value(details.get("Fuel"))
                row[header_map['26']] = engine_size_to_liter(clean_value(details.get("Engine size")))
                row[header_map['35']] = clean_value(details.get("CAP clean"))
                row[header_map['37']] = clean_value(details.get("CAP average"))
                row[header_map['16']] = clean_value(details.get("First reg"))
                row[header_map['21']] = clean_value(details.get("MOT"))
                row[header_map['18']] = clean_odometer(clean_value(details.get("Odometer")))
                row[header_map['19']] = clean_value(details.get("Mileage checked"))
                row[header_map['22']] = clean_value(details.get("Service history"))
                row[header_map['20']] = clean_value(details.get("V5 Location"))
                row[header_map['34']] = clean_value(details.get("Master key"))
                row[header_map['25']] = clean_value(details.get("Number of former keepers"))
                row[header_map['23']] = result["vat_status"] or ""
                row[header_map['13']] = result["location"] or ""
                row[header_map['29']] = Images or ""

 
                
                
                row[header_map['14']] = start_date or ""
                row[header_map['15']] = start_time or ""
                row[header_map['48']] = end_date or ""
                row[header_map['49']] = end_time or ""
                row[header_map['50']] = auction_type or ""
                row[header_map['51']] = details.get("Non-Runner", "") or ""
                row[header_map['52']] = [datas.get("current_bid", 0)]
                row[header_map['53']] = datas.get("current_bid", "") or 0          
      

            all_rows.append(row)
           


    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=headers)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\n✔ CSV Generated: {output_file}")


extract_manual_keys()


https://assets.wilsonsauctions.com///lots_1768295211-PICT0043.jpg_2086083/320x231.webp,https://assets.wilsonsauctions.com///lots_1768295211-PICT0044.jpg_1530300/320x231.webp,https://assets.wilsonsauctions.com///lots_1768295212-PICT0045.jpg_1906321/320x231.webp,https://assets.wilsonsauctions.com///lots_1768295212-PICT0046.jpg_1957969/320x231.webp,https://assets.wilsonsauctions.com///lots_1768295212-PICT0047.jpg_1541766/320x231.webp,https://assets.wilsonsauctions.com///lots_1768295213-PICT0048.jpg_1646647/320x231.webp,https://assets.wilsonsauctions.com///lots_1768295213-PICT0049.jpg_1814361/320x231.webp,https://assets.wilsonsauctions.com///lots_1768295214-PICT0050.jpg_2093501/320x231.webp,https://assets.wilsonsauctions.com///lots_1768295214-PICT0051.jpg_1954870/320x231.webp
https://assets.wilsonsauctions.com///lots_1768988815-P1050805.JPG_1580790/320x231.webp,https://assets.wilsonsauctions.com///lots_1768988815-P1050806.JPG_1547635/320x231.webp,https://assets.wilsonsauctions.com///lots_1

In [11]:
import re

def safe_int(value, default=""):

    try:
        if value is None:
            return default

        if isinstance(value, str):
            value = value.strip()
            if value == "":
                return default
            value = (
                value.replace("€", "")
                     .replace(",", "")
                     .replace(" ", "")
            )

        return int(float(value))
    except (ValueError, TypeError):
        return default
    

def to_int1(value, default=""):
    try:
        if value is None:
            return default
        value = str(value)
        value = value.encode("ascii", "ignore").decode()
        for ch in ["£", "€", ",", " "]:
            value = value.replace(ch, "")

        if value == "":
            return default

        return int(float(value))
    except Exception as e:

        return default


def to_int(value, default=""):
        try:
            return int(value)
        except (TypeError, ValueError):
            return default

def to_float(value):
        try:
            return float(value)
        except (TypeError, ValueError):
            return None


def getVarient(Variant,Derivative,auction_house):

    match auction_house:
        case "BCA" | "Aston Barclay":
            Val = Derivative
        case _:
             Val = Variant
    return Val

def FieldSet(data,auction_house):


    return {
        # Basic
        'title': data.get('Title'),
        "make_id": data.get("Make") or data.get("Manufacturer"),
        "model_id": data.get("Model"),
        "variant_id": getVarient(data.get("Variant") or data.get("variant"),data.get("Derivative") or data.get("derivative") ,auction_house)  ,
        'body_id': data.get('Body type') or data.get("Body Type"),
        'year': data.get('Year'),
        'center_id': data.get('Center'),
        'color': data.get('Colour'),
        'vin': data.get('VIN'),
        'lot': data.get('Lot'),

        # Vehicle Specs
        'doors' : to_int(data.get('doors') or data.get('Doors')),
        'seats': to_int(data.get('seats')  or data.get("Seats")),
        'fuel_type': data.get('Fuel Type'),
        'fuel_details': data.get('Fuel Type'),
        'transmission': data.get('Transmission'),
        'transmission_details': data.get('Transmission'),
        'euro_status': data.get('Euro Status') or data.get('Euro status'),
        'cc': data.get('CC'),
        'keys': data.get('Keys'),
        'engine_runs': data.get('Non Runner'),
        'mileage': data.get("Mileage",""),
        'mileage_warranted': data.get('Mileage Warranted',""),
        'former_keepers': data.get('Former Keepers', ""),
        'vat_status': data.get('VAT Status') or data.get("VAT status"),

        # Bidding & Pricing
        'bidding_history': data.get('Bidding History'),
        'last_bid': to_int1(
            data.get('Last Bid')
            or data.get('LAST BID')
            or data.get('Last bid')
            or data.get('last_bid')
            or data.get('Last Bid ')
        ),
        'bidding_status': data.get('Bidding Status') or data.get("bidding_status") or data.get('Bidding status'),
        'cap_new': safe_int(data.get('Cap New')),
        'cap_retail': safe_int(
            data.get('Cap Retail') or data.get('CAP retail')
        ),
        'cap_clean': safe_int(
            data.get('CAP Clean') or data.get('CAP clean')
        ),
        'cap_average': safe_int(data.get('CAP Average')),
        'cap_below': safe_int(data.get('CAP Below')),

        'glass_new': safe_int(data.get('Glass New')),
        'glass_retail': safe_int(data.get('Glass Retail')),
        'glass_trade': safe_int(data.get('Glass Trade')),
        'autotrader_retail_value': data.get('Autotrader Retail Value', ""),
        'autotrader_trade_value': data.get('Autotrader Trade Value', ""),
        'buy_now_price': data.get('buy_now_price'),

        # Dates
        'start_date': data.get('Start Date'),
        'start_time': data.get('Start Time'),
        'end_date': data.get('End Date'),
        'mot_expiry_date': data.get('MOT Expiry Date'),
        'mot_due': data.get('MOT Due'),
        'inspection_date': data.get('Inspection date'),
        'dor': data.get('D.O.R'),

        # Documents & Reports
        'v5': data.get('V5'),
        'reg': data.get('Reg') or data.get("reg"),
        'service_history': data.get('Service History'),
        'no_of_services': to_int(data.get('No of Service', "")),
        'number_of_services_details': data.get('number_of_services_details'),
        'last_service': data.get('Last Service'),
        'last_service_mileage': to_int(data.get('Last service mileage', "")),
        'dvsa_mileage': data.get('dvsa_mileage'),
        'inspection_report': data.get('Inspection Report'),
        'other_report': data.get('other_report'),
        'service_notes': data.get('Service Notes'),
        'vendor': data.get('vendor') or data.get("Vendor"),

        # Condition & Features
        'grade': to_int(data.get('Grade', "")),
        'tyres_condition': data.get('Tyres Condition'),
        'general_condition': data.get('General Condition'),
        'brakes': data.get('brakes'),
        'hubs': data.get('hubs'),
        'features': data.get('features'),
        'equipment': data.get('Equipment'),
        'additional_information': data.get('Additional information'),
        'imported': to_int(data.get('imported', "")),
        'declarations': data.get('declarations'),
        'damaged_images': data.get('Damaged_images'),
        'damage_details': data.get('Damage_details'),

        # Media
        'images': data.get('Images'),
    }



input_file = "final_time_wilson.csv"
output_file = "readytoupload_wilson.csv"


auction_house = "Wilsons Auctions"

with open(input_file, newline="", encoding="utf-8") as infile:
    reader = csv.DictReader(infile)
    fieldnames = None
    rows_to_write = []

    for row in reader:
        processed = FieldSet(row, auction_house)
        if not fieldnames:
       
            fieldnames = list(processed.keys())
        rows_to_write.append(processed)


with open(output_file, "w", newline="", encoding="utf-8") as outfile:
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows_to_write)

print(f"✅ Conversion complete: {len(rows_to_write)} rows written to {output_file}")

✅ Conversion complete: 127 rows written to readytoupload_wilson.csv


In [12]:
import requests
import json, os, csv
from datetime import datetime
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


with open(r"D:\bots\autoboli.json", "r", encoding="utf-8") as f:
    autoboli = json.load(f)

with open("auction_details.json","r",encoding="utf-8") as f:
    database = json.load(f)

baseurl = autoboli.get("url", "")
email = autoboli.get("email", "")
password = autoboli.get("password", "")


def get_base_folder_info():
    folder_name = os.path.basename(os.getcwd())
    parts = folder_name.split("-")
    if not parts or not parts[0].isdigit():
        return None, None
    sheet_id = parts[0]
    name_parts = parts[1:]
    if name_parts and name_parts[0].isdigit():
        name_parts = name_parts[1:]
    auction_name = "-".join(name_parts).strip()
    return sheet_id, auction_name


def loginAutoBoli():
    login_url = f"{baseurl}/api/auth/login"
    payload = {"email": email, "password": password}
    try:
        response = requests.post(login_url, json=payload)
        response.raise_for_status()  
        data = response.json()
        token = data.get("data", {}).get("token")
        if token:
            print("✅ Login successful")
            return token
        else:
            print("❌ Token not found in response")
            return None
    except requests.exceptions.RequestException as e:
        print("❌ Login failed:", e)
        return None


def getPlatefromID(auctionName="Wilsons Auctions", token=None):
    if not token:
        print("❌ No token provided")
        return None

    plate_url = f"{baseurl}api/cruds/platform"
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }

    try:
        response = requests.get(plate_url, headers=headers, verify=False, timeout=30)
        response.raise_for_status()
        data = response.json()
        plates = data.get("data", [])
        for plate in plates:
            if plate.get("name", "").strip() == auctionName:
                print(f"✅ Found {auctionName}: ID = {plate.get('id')}")
                return plate.get("id")
        print(f"❌ {auctionName} not found")
        return None
    except requests.exceptions.RequestException as e:
        print("❌ Failed to fetch plates:", e)
        return None


def uploadData(login_token, sheet_id, auction_name, auction_date, auction_type, platform_id, payload_rows, end_date=None):
    auctionUrl = f"{baseurl}/api/cruds/auctions"
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
        "Authorization": f"Bearer {login_token}"
    }


    data = {
        "id": sheet_id,
        "name": f"{auction_name} Upload",
        "auction_date": auction_date,
        "end_date": end_date,          
        "auction_type": auction_type,
        "platform_id": platform_id,
        "payload": payload_rows          
    }

    print("➡ Sending full payload:\n", data)
    try:
        response = requests.post(auctionUrl, data=json.dumps(data), headers=headers, verify=False, timeout=300)
        response.raise_for_status()
        print("✅ Upload successful")
        print(response.json())
    except requests.exceptions.RequestException as e:
        print("❌ Upload failed:", e)




class DataFomater:
    def __init__(self, data):
        self.data = data

    def Render(self, auction_house):
        body_id = self.data.get("body_id")
        return {
            'title': self.data.get('title'),
            'make_id': self.data.get("make_id"),
            'model_id': self.data.get("model_id"),
            'variant_id': "",
            'derivative': self.data.get("variant_id"),
            'body_id': body_id,
            'vehicle_id': None,
            'year': self.data.get('year'),
            'center_id': self.data.get('center_id'),
            'color': self.data.get('color'),
            'vin': self.data.get('vin'),
            'lot': self.data.get('lot'),
            'doors': self.data.get('doors'),
            'seats': self.data.get('seats'),
            'fuel_type': self.data.get('fuel_type'),
            'fuel_details': self.data.get('fuel_type'),
            'transmission': self.data.get('transmission'),
            'transmission_details': self.data.get('transmission'),
            'cc': self.data.get('cc'),
            'keys': self.data.get('keys'),
            'engine_runs': self.data.get('engine_runs'),
            'mileage': self.data.get("mileage", 0),
            'mileage_warranted': self.data.get('mileage_warranted'),
            'former_keepers': self.data.get('former_keepers'),
            'vat_status': self.data.get('vat_status'),
            'start_date': self.data.get('start_date'),
            'end_date': self.data.get('end_date'),
            'mot_expiry_date': self.data.get('mot_expiry_date'),
            'mot_due': self.data.get('mot_due'),
            'inspection_date': self.data.get('inspection_date'),
            'dor': self.data.get('dor'),
            'v5': self.data.get('v5'),
            'reg': self.data.get('reg'),
            'service_history': self.data.get('service_history'),
            'no_of_services': self.data.get('no_of_services'),
            'number_of_services_details': self.data.get('number_of_services_details'),
            'last_service': self.data.get('last_service'),
            'last_service_mileage': self.data.get('last_service_mileage'),
            'dvsa_mileage': self.data.get('dvsa_mileage'),
            'inspection_report': self.data.get('inspection_report'),
            'other_report': self.data.get('other_report'),
            'service_notes': self.data.get('service_notes'),
            'vendor': self.data.get('vendor'),
            'grade': self.data.get('grade', ""),
            'tyres_condition': self.data.get('tyres_condition'),
            'general_condition': self.data.get('general_condition'),
            'brakes': self.data.get('brakes'),
            'hubs': self.data.get('hubs'),
            'features': self.data.get('features'),
            'equipment': self.data.get('equipment'),
            'additional_information': self.data.get('additional_information'),
            'imported': self.data.get('imported'),
            'declarations': self.data.get('declarations'),
            'damaged_images': self.data.get('damaged_images'),
            'damage_details': self.data.get('damage_details'),
            'images': self.data.get('imagesss'),
        }


login_token = loginAutoBoli()
if login_token:
    auction_id = getPlatefromID("Wilsons Auctions", login_token)
    if auction_id:
        csv_path = "readytoupload_wilson.csv"
        rows = []

        auction_house = "Wilsons Auctions"

        with open(csv_path, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                formatter = DataFomater(row)
                mapped_row = formatter.Render(auction_house)
                rows.append(mapped_row)

        sheet_id, auction_name = get_base_folder_info()
        auction_date = database.get("start_date", "")
        endDate = database.get("end_date", "")
        try:
            auction_date = datetime.strptime(auction_date, "%Y-%m-%d").strftime("%d-%m-%Y")
            endDate = datetime.strptime(endDate, "%Y-%m-%d").strftime("%d-%m-%Y")
        except ValueError:
            pass

        payload = json.dumps(rows, ensure_ascii=False)
    
        uploadData(login_token, sheet_id, auction_name, auction_date, 2, auction_id, payload,endDate)

✅ Login successful
✅ Found Wilsons Auctions: ID = 9
➡ Sending full payload:
 {'id': '20', 'name': 'wilsonsauctions Upload', 'auction_date': '21-01-2026', 'end_date': '22-01-2026', 'auction_type': 2, 'platform_id': 9, 'payload': '[{"title": "BMW 118d SE", "make_id": "BMW", "model_id": "118d SE", "variant_id": "", "derivative": "SE", "body_id": "Hatchback", "vehicle_id": null, "year": "2006", "center_id": "Wilsons Auctions Maidstone", "color": "Black", "vin": "", "lot": "38", "doors": "5", "seats": "", "fuel_type": "Diesel", "fuel_details": "Diesel", "transmission": "Manual", "transmission_details": "Manual", "cc": "2.0", "keys": "Not present", "engine_runs": "No", "mileage": "", "mileage_warranted": "Not Warranted", "former_keepers": "", "vat_status": "Margin", "start_date": "2026-01-21", "end_date": "2026-01-22", "mot_expiry_date": "20-10-2026", "mot_due": "", "inspection_date": "", "dor": "08/09/2006", "v5": "Here", "reg": "AF56LHB", "service_history": "No", "no_of_services": "", "num